<a href="https://colab.research.google.com/github/Akpati-Lucan/algoverse-research/blob/master/Diffusion_Hebbian_BackProp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training a Denoising Diffusion Model on MNIST

In this notebook, we train a Denoising Diffusion Probabilistic Model (DDPM) using the DeepInverse library and the MNIST handwritten digits dataset.

A diffusion model learns to generate images by reversing a gradual noising process. During training, random Gaussian noise is added to images, and the neural network learns to predict the noise that was added. Once trained, this process can be reversed to generate entirely new images.

The main stages of this notebook are:



*   Import libraries
*   Load and preprocess the MNIST dataset
*   Build a diffusion U-Net
*   Define the diffusion noise schedule
*   Train the network
*   Save the trained model

# Importing the Required Libraries

We begin by importing the libraries needed throughout the notebook.

PyTorch provides tensor operations, GPU acceleration, automatic differentiation, and optimization routines.
DeepInverse supplies a pre-built diffusion U-Net architecture designed for inverse problems and diffusion modeling.
Torchvision provides convenient access to standard computer vision datasets and image transformations.

These libraries form the foundation of the training pipeline.

In [ ]:
!pip install deepinv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 30.9 MB/s eta 0:00:00


In [ ]:
import torch
import deepinv
from torchvision import datasets, transforms
import torch
import torch.nn.functional as F

# Configuring the Training Environment

The model is trained on the GPU whenever CUDA is available. GPU acceleration significantly reduces training time because thousands of image operations can be executed simultaneously.

Two important hyperparameters are also defined:

Batch size determines how many images are processed before updating the network.
Image size specifies the spatial dimensions of each training image. Although MNIST images are originally 28 x 28 pixels, they are resized to 32 x 32 to match the architecture used by the diffusion model.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

batch_size = 32
image_size = 32

# Image Preprocessing

Before images can be used for training, several preprocessing operations are applied.

Resize converts every image to 32×32 pixels.
ToTensor converts images into PyTorch tensors.
Normalize scales pixel values into a consistent numerical range, improving optimization stability.

These transformations ensure every training sample has the same format before entering the neural network.

In [ ]:
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.0,), (1.0,))
])

# Loading the MNIST Dataset

The MNIST dataset contains 60,000 grayscale images of handwritten digits (0-9).

The DataLoader performs several useful tasks:

Loads data in mini-batches
Randomly shuffles the images each epoch
Efficiently feeds images into the training loop

Mini-batch learning produces more stable optimization while making better use of GPU hardware.

In [ ]:
# Loading MNIST Dataset

full_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

# Use only a small subset for fast experiments
num_images = 1000

indices = torch.arange(num_images)

train_dataset = torch.utils.data.Subset(
    full_dataset,
    indices
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

print(f"Training images: {len(train_dataset)}")

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 525kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.45MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.97MB/s]

Training images: 1000


# Building the Diffusion Model

The neural network used in this notebook is a Diffusion U-Net.

A U-Net consists of:

an encoder that extracts increasingly abstract image features,
a bottleneck that captures global information,
a decoder that reconstructs the desired output.

Instead of generating images directly, the network learns to estimate the random Gaussian noise added to each image.

The Adam optimizer updates the model parameters after every mini-batch, while Mean Squared Error (MSE) measures how accurately the predicted noise matches the true noise.

In [ ]:
lr = 1e-4
epochs = 20

model = deepinv.models.DiffUNet(
    in_channels=1,
    out_channels=1,
    pretrained=None
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

mse = torch.nn.MSELoss()

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Find the convolutional layers

In [ ]:
print("Convolutional layers:")

for name, module in model.named_modules():
    if isinstance(module, torch.nn.Conv2d):
        print(name, module)

Capture the activation entering that layer

In [ ]:
hebbian_input = None


def save_hebbian_input(module, inputs, output):
    global hebbian_input
    hebbian_input = inputs[0].detach()


hebbian_hook = hebbian_layer.register_forward_hook(
    save_hebbian_input
)

# Constructing the Diffusion Noise Schedule

Diffusion models gradually corrupt images over many time steps.

The parameter β (beta) determines how much noise is added at each step.

Using these values, we compute:

* α (alpha): the amount of original image preserved.
* Cumulative alpha: the fraction of signal remaining after many diffusion steps.
* Square-root terms: constants used to efficiently generate noisy training examples.

These values implement the forward diffusion equation:

xt = sqrt(αt)x0 + sqrt(1−αt)ϵ

where

* x0 is the clean image,
* xt is the noisy image,
* ϵ is Gaussian noise.

In [ ]:
beta_start = 1e-4
beta_end = 0.02
timesteps = 1000

betas = torch.linspace(beta_start, beta_end, timesteps, device=device)

alphas = 1.0 - betas

alphas_cumprod = torch.cumprod(alphas, dim=0)

sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)

sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

In [ ]:
import torch
import torch.nn.functional as F


def hebbian_conv_update(
    layer,
    layer_input,
    local_error,
    learning_rate
):

    # Extract input patches corresponding to each convolution
    patches = F.unfold(
        layer_input,
        kernel_size=layer.kernel_size,
        dilation=layer.dilation,
        padding=layer.padding,
        stride=layer.stride
    )

    # [B, Cin*K*K, L]

    error = local_error.flatten(2)

    # Make sure the number of spatial locations agrees
    if patches.shape[-1] != error.shape[-1]:
        raise ValueError(
            f"Spatial mismatch: "
            f"patches={patches.shape}, "
            f"error={error.shape}"
        )

    # Hebbian update:
    #
    # ΔW = η * pre-synaptic activity * error
    #
    update = torch.einsum(
        "bol,bil->oi",
        error,
        patches
    )

    # Average across batch and spatial positions
    update /= (
        layer_input.size(0) *
        patches.size(2)
    )

    update = update.view_as(layer.weight)

    # Normalize update
    update = update / (
        update.norm() + 1e-8
    )

    with torch.no_grad():
        layer.weight.add_(
            learning_rate * update
        )

# Training the Diffusion Model

Training follows the standard DDPM algorithm.

For every mini-batch:

A batch of clean images is loaded.
Random Gaussian noise is generated.
A random diffusion timestep is selected.
Noise is added according to the diffusion schedule.
The noisy image and timestep are passed into the U-Net.
The network predicts the noise.
The prediction is compared with the true noise using MSE loss.
Backpropagation computes gradients.
Adam updates the model parameters.

Unlike traditional image classification, the objective is not to predict labels but to accurately estimate the noise added to each image

In [ ]:
hebbian_lr = 1e-4

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for batch_idx, (data, _) in enumerate(train_loader):

        imgs = data.to(device)

        # -------------------------
        # Sample Gaussian noise
        # -------------------------

        noise = torch.randn_like(imgs)

        # -------------------------
        # Random timestep
        # -------------------------

        t = torch.randint(
            0,
            timesteps,
            (imgs.size(0),),
            device=device
        )

        # -------------------------
        # Forward diffusion
        # -------------------------

        noised_imgs = (
            sqrt_alphas_cumprod[
                t, None, None, None
            ] * imgs
            +
            sqrt_one_minus_alphas_cumprod[
                t, None, None, None
            ] * noise
        )

        # -------------------------
        # Forward pass
        # -------------------------

        optimizer.zero_grad()

        estimated_noise = model(
            noised_imgs,
            t,
            type_t="timestep"
        )

        # -------------------------
        # MSE
        # -------------------------

        loss = mse(
            estimated_noise,
            noise
        )

        # -------------------------
        # Backpropagation
        # -------------------------
        loss.backward()
        optimizer.step()

        # -------------------------
        # Hebbian learning
        # -------------------------

        local_error = (
            noise -
            estimated_noise.detach()
        )

        hebbian_conv_update(
            layer=hebbian_layer,
            layer_input=hebbian_input,
            local_error=local_error,
            learning_rate=hebbian_lr
        )

        total_loss += loss.item()

    avg_loss = (
        total_loss /
        len(train_loader)
    )

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {avg_loss:.5f}"
    )

Epoch [1/20] Loss: 0.53875
Epoch [2/20] Loss: 0.06774
Epoch [3/20] Loss: 0.02539
Epoch [4/20] Loss: 0.01968
Epoch [5/20] Loss: 0.02344
Epoch [6/20] Loss: 0.02129
Epoch [7/20] Loss: 0.01862
Epoch [8/20] Loss: 0.01766
Epoch [9/20] Loss: 0.01668
Epoch [10/20] Loss: 0.01515
Epoch [11/20] Loss: 0.01656
Epoch [12/20] Loss: 0.01606
Epoch [13/20] Loss: 0.01593
Epoch [14/20] Loss: 0.01567
Epoch [15/20] Loss: 0.01435
Epoch [16/20] Loss: 0.01671
Epoch [17/20] Loss: 0.01470
Epoch [18/20] Loss: 0.01549
Epoch [19/20] Loss: 0.01442
Epoch [20/20] Loss: 0.01578


# Saving the Trained Model

After training is complete, the learned network parameters are saved to disk.

The saved checkpoint can later be loaded for:

image generation,
fine-tuning,
evaluation,
inference without retraining.

Saving only the model's state_dict() is the standard PyTorch practice because it stores the learned weights while keeping the file size relatively small.

In [ ]:
torch.save(model.state_dict(), "trained_diffusion_model.pth")